# 03 - Regras de associação (Apriori)
**Objetivo:** encontrar combinações de condições (causa, tipo, clima, pista...) que **aparecem junto** com acidentes com vítima fatal.
Atenção: regras mostram **associação**, não causa. As variáveis das regras são diferentes das usadas no agrupamento.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = RAIZ / "data"; FIG = RAIZ / "figuras"; FIG.mkdir(exist_ok=True)
ANO = 2023
SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

from mlxtend.frequent_patterns import apriori, association_rules
base = pd.read_pickle(DATA / "base_acidentes.pkl")
print(base.shape)

## Pré-processamento
Só variáveis categóricas, **fora** das usadas no agrupamento (pessoas, veículos, mortos, feridos). Categorias raras (<1%) viram "Outros" para não gerar itens com suporte minúsculo. Cada par coluna=valor vira uma coluna 0/1 (one-hot).

In [ ]:
CAT = ["causa_acidente", "tipo_acidente", "fase_dia", "condicao_metereologica",
       "tipo_pista", "tracado_via", "uso_solo", "dia_semana"]       # NÃO CONFIRMADO: confira nomes
CAT = [c for c in CAT if c in base.columns]
print("Usando:", CAT)
alvo_col = "classificacao_acidente"
assert alvo_col in base.columns, "Coluna de classificação não encontrada (NÃO CONFIRMADO)."
print(base[alvo_col].value_counts())

d = base[CAT].copy()
for c in CAT:
    freq = d[c].value_counts(normalize=True)
    raras = freq[freq < 0.01].index
    d[c] = d[c].where(~d[c].isin(raras), "Outros").fillna("Ignorado")
d["FATAL"] = base[alvo_col].str.contains("fatais", case=False, na=False)
onehot = pd.get_dummies(d.drop(columns="FATAL"), prefix_sep="=").astype(bool)
onehot["FATAL"] = d["FATAL"]
print("Matriz:", onehot.shape, "| % fatais:", round(d["FATAL"].mean()*100, 2))

## Apriori - testando suportes mínimos entre 1% e 5%

In [ ]:
for s in [0.05, 0.03, 0.02, 0.01]:
    f = apriori(onehot, min_support=s, use_colnames=True, max_len=4)
    print(f"min_support={s:.0%}: {len(f)} conjuntos frequentes")

In [ ]:
MIN_SUP = 0.01   # <- AJUSTE com base na saída acima (explique a escolha no relatório)
freq = apriori(onehot, min_support=MIN_SUP, use_colnames=True, max_len=4)
try:
    regras = association_rules(freq, metric="confidence", min_threshold=0.10, num_itemsets=len(onehot))
except TypeError:
    regras = association_rules(freq, metric="confidence", min_threshold=0.10)
# Consequente fixo: acidente com vítima fatal
foco = regras[regras["consequents"].apply(lambda s: s == frozenset({"FATAL"}))].copy()
foco["antecedente"] = foco["antecedents"].apply(lambda s: " + ".join(sorted(s)))
foco = foco.sort_values("lift", ascending=False)
print("Regras com consequente FATAL:", len(foco), "| base de fatais (confiança esperada ao acaso):", round(onehot['FATAL'].mean(), 3))
display(foco[["antecedente", "support", "confidence", "lift"]].head(20).round(3))

Como ler: **suporte** = fração de TODOS os acidentes em que antecedente e fatal ocorrem juntos; **confiança** = entre os acidentes com o antecedente, fração fatal; **lift** > 1 = fatal aparece mais do que ao acaso com aquelas condições.

In [ ]:
display(foco.sort_values("support", ascending=False)[["antecedente", "support", "confidence", "lift"]].head(10).round(3))
display(foco[foco["lift"] > 1].sort_values("confidence", ascending=False)[["antecedente", "support", "confidence", "lift"]].head(10).round(3))
foco[["antecedente", "support", "confidence", "lift"]].round(4).to_csv(RAIZ / "figuras" / "regras_fatal.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 5))
sc = plt.scatter(foco["support"], foco["confidence"], c=foco["lift"], cmap="viridis", s=25, alpha=.8)
plt.colorbar(sc, label="lift"); plt.xlabel("Suporte"); plt.ylabel("Confiança")
plt.title("Regras → acidente com vítima fatal"); plt.tight_layout()
plt.savefig(FIG / "regras_suporte_confianca.png", dpi=150); plt.show()

## Discussão (escrever)
Comente as 3-5 regras mais interessantes. Lembre: associação, não causa; regras com suporte baixo podem ser casuais; compare com a taxa geral de fatais.